# Model Performance Analysis - Interactive Exploration

This notebook provides interactive exploration of which power plants are predicted more accurately and why.

## Key Findings Summary:

1. **Best Predicted Plants**: Large coal-fired plants in Western U.S. (Navajo, Hunter, Intermountain)
2. **Most Important Feature**: **NOx emissions** (3x more important than next feature)
3. **Critical Pattern**: High performers have **2.4x higher NOx emissions** (872 vs 259 lbs)
4. **Geographic Pattern**: Higher altitude plants predict better
5. **Key Correlations**: 
   - Recall vs NOx Mass: r = 0.654 (very strong)
   - F1 vs NOx Mass: r = 0.625 (strong)
   - Precision vs NOx Mass: r = 0.527 (moderate)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load results
results_dir = '/net/fs06/d3/rzhuang/TROPOMI_US/results/'
plant_metrics = pd.read_csv(f'{results_dir}per_plant_performance_metrics.csv')

with open(f'{results_dir}feature_importance_gradient.json', 'r') as f:
    feature_importance = json.load(f)

with open(f'{results_dir}feature_comparison_high_vs_low_performers.json', 'r') as f:
    feature_comparison = json.load(f)

print(f"Loaded data for {len(plant_metrics)} power plants")

## 1. Overall Performance Distribution

In [ ]:
# Summary statistics
print("Performance Metrics Summary:")
print("="*60)
print(plant_metrics[['accuracy', 'precision', 'recall', 'f1', 'auc']].describe())

# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(plant_metrics['f1'], bins=30, edgecolor='black')
axes[0].axvline(plant_metrics['f1'].mean(), color='red', linestyle='--', label='Mean')
axes[0].set_title('F1 Score Distribution')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(plant_metrics['auc'], bins=30, edgecolor='black')
axes[1].axvline(plant_metrics['auc'].mean(), color='red', linestyle='--', label='Mean')
axes[1].set_title('AUC Distribution')
axes[1].set_xlabel('AUC')
axes[1].set_ylabel('Count')
axes[1].legend()

axes[2].hist(plant_metrics['accuracy'], bins=30, edgecolor='black')
axes[2].axvline(plant_metrics['accuracy'].mean(), color='red', linestyle='--', label='Mean')
axes[2].set_title('Accuracy Distribution')
axes[2].set_xlabel('Accuracy')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Best and Worst Predicted Plants

In [ ]:
# Top 10 best
print("\nTOP 10 BEST PREDICTED PLANTS:")
print("="*80)
best_plants = plant_metrics.nlargest(10, 'f1')
print(best_plants[['Facility_ID', 'Facility_Name', 'f1', 'accuracy', 'auc', 
                   'n_observations', 'Total_NOx_Mass']].to_string(index=False))

# Top 10 worst
print("\n\nTOP 10 WORST PREDICTED PLANTS:")
print("="*80)
worst_plants = plant_metrics.nsmallest(10, 'f1')
print(worst_plants[['Facility_ID', 'Facility_Name', 'f1', 'accuracy', 'auc', 
                    'n_observations', 'Total_NOx_Mass']].to_string(index=False))

## 3. Feature Importance Analysis

In [ ]:
# Sort features by importance
sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:15]
features, importances = zip(*sorted_features)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(features))
ax.barh(y_pos, importances, alpha=0.8, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(features)
ax.invert_yaxis()
ax.set_xlabel('Importance (Gradient Magnitude)')
ax.set_title('Top 15 Most Important Features')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTOP 10 MOST IMPORTANT FEATURES:")
for i, (feat, imp) in enumerate(sorted_features[:10], 1):
    print(f"{i:2d}. {feat:<45s}: {imp:.6f}")

## 4. High vs Low Performers - Feature Differences

In [ ]:
# Sort by relative difference
sorted_diff = sorted(feature_comparison.items(), 
                    key=lambda x: abs(x[1]['relative_difference']), 
                    reverse=True)[:10]

print("\nFEATURES WITH LARGEST DIFFERENCES (High vs Low Performers):")
print("="*80)
for i, (feature, stats) in enumerate(sorted_diff, 1):
    print(f"\n{i:2d}. {feature}")
    print(f"    High performers: {stats['high_performers_mean']:.4f}")
    print(f"    Low performers:  {stats['low_performers_mean']:.4f}")
    print(f"    Rel. difference: {stats['relative_difference']:+.2f}%")

# Visualize
features_diff = [f[0] for f in sorted_diff]
rel_diffs = [f[1]['relative_difference'] for f in sorted_diff]
colors = ['green' if x > 0 else 'red' for x in rel_diffs]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(features_diff))
ax.barh(y_pos, rel_diffs, alpha=0.8, color=colors, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(features_diff)
ax.invert_yaxis()
ax.set_xlabel('Relative Difference (%)')
ax.set_title('Feature Differences: High vs Low Performers\n(Green = Higher in High Performers)')
ax.axvline(0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Load correlation results
corr_df = pd.read_csv(f'{results_dir}correlation_analysis.csv')

# Filter significant correlations
significant = corr_df[
    (corr_df['pearson_p'] < 0.05) & 
    (abs(corr_df['pearson_r']) > 0.3)
].sort_values('pearson_r', key=abs, ascending=False)

print("\nSIGNIFICANT CORRELATIONS (p < 0.05, |r| > 0.3):")
print("="*80)
for _, row in significant.iterrows():
    print(f"{row['performance_metric']:<12s} vs {row['plant_characteristic']:<30s}: "
          f"r = {row['pearson_r']:+.3f} (p={row['pearson_p']:.4f})")

## 6. NOx Emissions vs Performance (Key Relationship)

In [ ]:
# Filter valid data
valid_data = plant_metrics[plant_metrics['Total_NOx_Mass'].notna()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F1 vs NOx
axes[0].scatter(valid_data['Total_NOx_Mass'], valid_data['f1'], 
                alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Total NOx Mass')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score vs NOx Emissions')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Recall vs NOx
axes[1].scatter(valid_data['Total_NOx_Mass'], valid_data['recall'], 
                alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Total NOx Mass')
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall vs NOx Emissions')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)

# Precision vs NOx
axes[2].scatter(valid_data['Total_NOx_Mass'], valid_data['precision'], 
                alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Total NOx Mass')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision vs NOx Emissions')
axes[2].set_xscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlations
print("\nCorrelations with NOx Emissions:")
print(f"F1 Score:  r = {valid_data[['Total_NOx_Mass', 'f1']].corr().iloc[0,1]:.3f}")
print(f"Recall:    r = {valid_data[['Total_NOx_Mass', 'recall']].corr().iloc[0,1]:.3f}")
print(f"Precision: r = {valid_data[['Total_NOx_Mass', 'precision']].corr().iloc[0,1]:.3f}")
print(f"AUC:       r = {valid_data[['Total_NOx_Mass', 'auc']].corr().iloc[0,1]:.3f}")

## 7. Geographic Patterns

In [ ]:
# Filter valid location data
geo_data = plant_metrics[plant_metrics['Latitude'].notna() & plant_metrics['Longitude'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score map
scatter1 = axes[0].scatter(geo_data['Longitude'], geo_data['Latitude'],
                          c=geo_data['f1'], s=100, cmap='RdYlGn', 
                          vmin=0, vmax=1, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('F1 Score by Geographic Location')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='F1 Score')

# AUC map
scatter2 = axes[1].scatter(geo_data['Longitude'], geo_data['Latitude'],
                          c=geo_data['auc'], s=100, cmap='RdYlGn', 
                          vmin=0.5, vmax=1, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('AUC by Geographic Location')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='AUC')

plt.tight_layout()
plt.show()

# Regional analysis
print("\nPerformance by Region (longitude-based):")
geo_data['region'] = pd.cut(geo_data['Longitude'], 
                            bins=[-130, -110, -100, -90, -70],
                            labels=['West', 'Mountain', 'Central', 'East'])
print(geo_data.groupby('region')[['f1', 'auc', 'accuracy']].mean())

## 8. Search for Specific Plant

In [ ]:
# Search by plant name or ID
search_term = "Navajo"  # Change this to search for different plants

results = plant_metrics[
    plant_metrics['Facility_Name'].str.contains(search_term, case=False, na=False) |
    plant_metrics['Facility_ID'].astype(str).str.contains(search_term, case=False)
]

if len(results) > 0:
    print(f"\nFound {len(results)} plant(s) matching '{search_term}':")
    print("="*80)
    for _, plant in results.iterrows():
        print(f"\nPlant: {plant['Facility_Name']} (ID: {plant['Facility_ID']})")
        print(f"  Location: {plant['State']}, ({plant['Latitude']:.2f}, {plant['Longitude']:.2f})")
        print(f"  Performance:")
        print(f"    F1 Score:  {plant['f1']:.4f}")
        print(f"    Accuracy:  {plant['accuracy']:.4f}")
        print(f"    Recall:    {plant['recall']:.4f}")
        print(f"    Precision: {plant['precision']:.4f}")
        print(f"    AUC:       {plant['auc']:.4f}")
        print(f"  Observations: {plant['n_observations']}")
        if pd.notna(plant['Total_NOx_Mass']):
            print(f"  NOx Emissions: {plant['Total_NOx_Mass']:.2f}")
else:
    print(f"No plants found matching '{search_term}'")

## Summary of Key Insights

### What Makes a Plant Easy to Predict?

1. **High NOx Emissions** (most important factor)
   - Best performers: 872 lbs average
   - Worst performers: 259 lbs average
   - 2.4x difference!

2. **Geographic/Environmental Factors**
   - Higher altitude (cleaner background)
   - Western U.S. location
   - Lower aerosol loading
   - Less water vapor

3. **Observation Quality**
   - Better viewing geometry
   - Less cloud interference
   - Higher surface albedo

### Recommendations:

1. **For better predictions of low-emission plants:**
   - Use emission-stratified models
   - Apply class weights based on emission levels
   - Enhance atmospheric correction

2. **For model improvement:**
   - Focus training on medium-emission plants
   - Add regional-specific features
   - Use ensemble methods